# NB 05 — Registros SUNAT de Compras y Ventas
**Proyecto 5 · Automatización Contable**

**Input:** `data/processed/facturas_extraidas.json`
*(Prueba multi-empresa 2026-07-21: se omite NB02/validación SUNAT — no depende de Clave SOL, pero se salta por decisión explícita para esta corrida.)*

**Output:**
- `data/output/Registro_Compras_{periodo}.xlsx` — Formato SUNAT 8.1 (single-empresa, cliente `CLIENTE_ID`)
- `data/output/Registro_Ventas_{ruc_emisor}_{periodo}.xlsx` — Formato SUNAT 14.1, **un archivo por empresa emisora y por mes (periodo)** detectado en el lote de ventas

No requiere plan de cuentas — rubro-agnóstico. Clasificación tributaria por regla determinista (gravada por defecto, excepciones a revisión manual).

## 0. Setup

In [1]:
import json
import openpyxl
from openpyxl.styles import PatternFill
from pathlib import Path
from datetime import datetime, date
from dotenv import load_dotenv
import cliente_config
from convertir_plantilla_sunat import convertir_si_falta

load_dotenv(dotenv_path=Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env'))

BASE_DIR    = Path('../')
CONFIG_DIR  = BASE_DIR / 'config'
# Prueba multi-empresa: se lee directo de facturas_extraidas.json (NB02 omitida)
INPUT_PATH  = BASE_DIR / 'data/processed/facturas_extraidas.json'
OUTPUT_DIR  = BASE_DIR / 'data/output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CLIENTE_ID / CLIENTE solo se usan para el Registro de Compras (single-empresa).
# El Registro de Ventas ya no depende de esto: agrupa por ruc_emisor de cada factura.
CLIENTE_ID = 'demo_pitch'
CLIENTE = cliente_config.cargar_cliente(CLIENTE_ID, CONFIG_DIR)

PLANTILLA_COMPRAS = convertir_si_falta(
    BASE_DIR / 'config/234_formato81_compra.xls',
    CONFIG_DIR / 'planes/formato81_compra.xlsx'
)
PLANTILLA_VENTAS = convertir_si_falta(
    BASE_DIR / 'config/234_formato141_venta.xls',
    CONFIG_DIR / 'planes/formato141_venta.xlsx'
)

FILL_AMARILLO = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')

print(f'Cliente activo (solo Compras): {CLIENTE["nombre"]}')
print(f'RUC: {CLIENTE.get("ruc")}  Razon social: {CLIENTE.get("razon_social")}')
print(f'Plantilla compras: {PLANTILLA_COMPRAS}')
print(f'Plantilla ventas: {PLANTILLA_VENTAS}')

Cliente activo (solo Compras): EMPRESA DEMO
RUC: None  Razon social: EMPRESA DEMO
Plantilla compras: ..\config\planes\formato81_compra.xlsx
Plantilla ventas: ..\config\planes\formato141_venta.xlsx


## 1. Helpers (fecha, redondeo, correlativo, dedup)

Nota: las plantillas oficiales solo traen ~10 filas en blanco entre el encabezado y la fila de TOTALES/notas al pie. Si un periodo real supera esa cantidad de comprobantes, escribir_fila_* sobreescribiría la fila de TOTALES — fuera de alcance por ahora (el piloto actual tiene 1-4 comprobantes por periodo).

In [2]:
def parse_fecha(fecha_str) -> datetime:
    if isinstance(fecha_str, datetime):
        return fecha_str
    if isinstance(fecha_str, date):
        return datetime(fecha_str.year, fecha_str.month, fecha_str.day)
    try:
        return datetime.strptime(str(fecha_str)[:10], '%Y-%m-%d')
    except Exception:
        return None


def round2(val) -> float:
    try:
        return round(float(val or 0), 2)
    except (ValueError, TypeError):
        return 0.0


def get_ultimo_correlativo(ws, fila_inicio_datos: int) -> int:
    maximo = 0
    for row in ws.iter_rows(min_row=fila_inicio_datos, values_only=True):
        val = row[0]
        if isinstance(val, (int, float)) and val > maximo:
            maximo = int(val)
    return maximo


def get_primera_fila_libre(ws, fila_inicio_datos: int) -> int:
    """Primera fila vacia (columna correlativo=None) a partir de fila_inicio_datos.
    NO usar ws.max_row: la plantilla ya trae filas de TOTALES y notas al pie
    mas abajo, que ws.max_row si cuenta."""
    fila = fila_inicio_datos
    while ws.cell(row=fila, column=1).value is not None:
        fila += 1
    return fila


def _fecha_str(valor) -> str:
    if hasattr(valor, 'strftime'):
        return valor.strftime('%Y-%m-%d')
    return str(valor or '')[:10]


def clave_documento(serie, numero, total, fecha) -> str:
    """Clave de deduplicacion: serie+numero+total+fecha.
    Incluir total y fecha evita tratar como 'ya registrado' a un comprobante
    genuinamente distinto cuyo numero se leyo mal en la extraccion (numero
    solo no es confiable como identificador unico)."""
    total_str = f'{round2(total):.2f}'
    return f'{serie}-{numero}-{total_str}-{_fecha_str(fecha)}'


def leer_docs_existentes(ws, fila_inicio_datos: int, col_serie: int, col_numero: int, col_total: int) -> set:
    existentes = set()
    for row in ws.iter_rows(min_row=fila_inicio_datos, values_only=True):
        serie = row[col_serie - 1]
        numero = row[col_numero - 1]
        if serie or numero:
            existentes.add(clave_documento(serie, numero, row[col_total - 1], row[1]))
    return existentes


def parsear_referencia(doc_referencia) -> tuple[str, str]:
    if not doc_referencia or '-' not in str(doc_referencia):
        return '', ''
    serie, numero = str(doc_referencia).split('-', 1)
    return serie.strip(), numero.strip()


def construir_lookup_referencias(facturas: list[dict]) -> dict:
    lookup = {}
    for f in facturas:
        clave = (str(f.get('serie', '')).strip(), str(f.get('numero', '')).strip())
        lookup[clave] = {
            'fecha_emision': f.get('fecha_emision'),
            'codigo_tipo_doc': f.get('codigo_tipo_doc', '01'),
        }
    return lookup

print('Helpers OK')

Helpers OK


## 2. Clasificación tributaria

In [3]:
TIPOS_DOC_COMUNES = {'FACTURA', 'BOLETA'}


def clasificar_tributario(factura: dict) -> dict:
    """
    Regla determinista: gravada por defecto si hay IGV y el comprobante es
    de un tipo comun (Factura/Boleta). NC/ND y cualquier otro caso quedan
    marcados para revision manual (no se adivina exonerada/inafecta/no gravada).
    """
    igv = float(factura.get('igv') or 0)
    tipo_doc = factura.get('tipo_doc', '')

    if igv > 0 and tipo_doc in TIPOS_DOC_COMUNES:
        return {'tratamiento_tributario': 'GRAVADA', 'requiere_revision_tributaria': False}

    return {'tratamiento_tributario': None, 'requiere_revision_tributaria': True}


def sospecha_emisor_receptor_invertido(factura: dict) -> bool:
    """
    Regla determinista de seguridad -- NO depende de que Claude acierte.
    Un RUC que empieza con '1' es persona natural; uno que empieza con '2' es
    persona juridica (empresa). El patron emisor=persona natural + receptor=
    empresa es el que se demostro (sesion de debugging 2026-07-14, 3 intentos
    de prompt distintos, todos fallidos) que la extraccion confunde de forma
    sistematica. No se corrige el dato solo -- se fuerza revision manual.
    El patron inverso (emisor=empresa, receptor=persona natural) es el caso
    normal y comun (empresa vendiendo a un cliente particular) y NO se marca.
    """
    ruc_emisor = str(factura.get('ruc_emisor') or '')
    ruc_receptor = str(factura.get('ruc_receptor') or '')
    return ruc_emisor.startswith('1') and ruc_receptor.startswith('2')

print('Clasificacion tributaria lista')

Clasificacion tributaria lista


## 3. Registro de Compras (Formato 8.1)

In [4]:
def escribir_fila_compra(ws, fila: int, correlativo: int, factura: dict, lookup_ref: dict) -> None:
    tratamiento = factura.get('tratamiento_tributario')
    es_ajuste = factura.get('tipo_doc') in ('NOTA_CREDITO', 'NOTA_DEBITO')

    base = round2(factura.get('base_imponible'))
    igv = round2(factura.get('igv'))
    total = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv = round2(total - base)

    ws.cell(row=fila, column=1, value=correlativo)
    ws.cell(row=fila, column=2, value=parse_fecha(factura.get('fecha_emision')))
    ws.cell(row=fila, column=3, value=parse_fecha(factura.get('fecha_vencimiento_pago')))
    ws.cell(row=fila, column=4, value=factura.get('codigo_tipo_doc'))
    ws.cell(row=fila, column=5, value=factura.get('serie'))
    # columna 6 (año DUA) queda vacia — fuera de alcance
    ws.cell(row=fila, column=7, value=factura.get('numero'))
    ws.cell(row=fila, column=8, value='6')  # proveedor siempre RUC
    ws.cell(row=fila, column=9, value=factura.get('ruc_emisor'))
    ws.cell(row=fila, column=10, value=factura.get('razon_social_emisor'))

    if tratamiento == 'GRAVADA':
        ws.cell(row=fila, column=11, value=base)
        ws.cell(row=fila, column=12, value=igv)

    ws.cell(row=fila, column=20, value=total)

    if factura.get('moneda') == 'USD' and factura.get('tipo_cambio'):
        ws.cell(row=fila, column=24, value=factura.get('tipo_cambio'))

    if es_ajuste:
        ref_serie, ref_numero = parsear_referencia(factura.get('doc_referencia'))
        ref_info = lookup_ref.get((ref_serie, ref_numero), {})
        ws.cell(row=fila, column=25, value=parse_fecha(ref_info.get('fecha_emision')))
        ws.cell(row=fila, column=26, value=ref_info.get('codigo_tipo_doc', '01'))
        ws.cell(row=fila, column=27, value=ref_serie)
        ws.cell(row=fila, column=28, value=ref_numero)

    if factura.get('requiere_revision_tributaria', True):
        for col in range(1, 29):
            ws.cell(row=fila, column=col).fill = FILL_AMARILLO

print('escribir_fila_compra OK')

escribir_fila_compra OK


In [5]:
with open(INPUT_PATH, encoding='utf-8') as f:
    todas_facturas = json.load(f)

for f in todas_facturas:
    f.update(clasificar_tributario(f))

compras = sorted([f for f in todas_facturas if f.get('tipo_operacion') == 'COMPRA'],
                  key=lambda x: x.get('fecha_emision', ''))
lookup_ref = construir_lookup_referencias(todas_facturas)

todas_fechas = [f.get('fecha_emision', '') for f in todas_facturas if f.get('fecha_emision')]
periodo = datetime.now().strftime('%Y%m') if not todas_fechas else todas_fechas[0][:7].replace('-', '')

output_compras = OUTPUT_DIR / f'Registro_Compras_{periodo}.xlsx'
wb_compras = openpyxl.load_workbook(str(PLANTILLA_COMPRAS))
ws_compras = wb_compras.active

FILA_INICIO_COMPRAS = 14
ultimo_correlativo = get_ultimo_correlativo(ws_compras, FILA_INICIO_COMPRAS)
docs_existentes = leer_docs_existentes(ws_compras, FILA_INICIO_COMPRAS, col_serie=5, col_numero=7, col_total=20)

fila_actual = get_primera_fila_libre(ws_compras, FILA_INICIO_COMPRAS)
correlativo = ultimo_correlativo + 1
escritas = 0
ignoradas = 0

for f in compras:
    clave = clave_documento(f.get('serie', ''), f.get('numero', ''), f.get('total'), f.get('fecha_emision'))
    if clave in docs_existentes:
        print(f'  Duplicado ignorado: {clave}')
        ignoradas += 1
        continue
    escribir_fila_compra(ws_compras, fila_actual, correlativo, f, lookup_ref)
    docs_existentes.add(clave)
    fila_actual += 1
    correlativo += 1
    escritas += 1

# RUC/razon social/periodo en el encabezado
ws_compras['B3'] = periodo
ws_compras['B4'] = CLIENTE.get('ruc')
ws_compras['B5'] = CLIENTE.get('razon_social')

wb_compras.save(str(output_compras))
print(f'Registro de Compras guardado: {output_compras}')
print(f'Filas escritas: {escritas}  Duplicados ignorados: {ignoradas}')

Registro de Compras guardado: ..\data\output\Registro_Compras_202406.xlsx
Filas escritas: 0  Duplicados ignorados: 0


## 4. Registro de Ventas (Formato 14.1)

In [6]:
def escribir_fila_venta(ws, fila: int, correlativo: int, factura: dict, lookup_ref: dict) -> None:
    tratamiento = factura.get('tratamiento_tributario')
    es_ajuste = factura.get('tipo_doc') in ('NOTA_CREDITO', 'NOTA_DEBITO')

    base = round2(factura.get('base_imponible'))
    igv = round2(factura.get('igv'))
    total = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv = round2(total - base)

    ws.cell(row=fila, column=1, value=correlativo)
    ws.cell(row=fila, column=2, value=parse_fecha(factura.get('fecha_emision')))
    ws.cell(row=fila, column=3, value=parse_fecha(factura.get('fecha_vencimiento_pago')))
    ws.cell(row=fila, column=4, value=factura.get('codigo_tipo_doc'))
    ws.cell(row=fila, column=5, value=factura.get('serie'))
    ws.cell(row=fila, column=6, value=factura.get('numero'))
    ws.cell(row=fila, column=7, value=factura.get('tipo_doc_identidad_receptor'))
    numero_doc_id = factura.get('numero_doc_identidad_receptor') or factura.get('ruc_receptor')
    ws.cell(row=fila, column=8, value=numero_doc_id)
    ws.cell(row=fila, column=9, value=factura.get('razon_social_receptor'))

    if tratamiento == 'GRAVADA':
        ws.cell(row=fila, column=11, value=base)
        ws.cell(row=fila, column=15, value=igv)

    ws.cell(row=fila, column=17, value=total)

    if factura.get('moneda') == 'USD' and factura.get('tipo_cambio'):
        ws.cell(row=fila, column=18, value=factura.get('tipo_cambio'))

    if es_ajuste:
        ref_serie, ref_numero = parsear_referencia(factura.get('doc_referencia'))
        ref_info = lookup_ref.get((ref_serie, ref_numero), {})
        ws.cell(row=fila, column=19, value=parse_fecha(ref_info.get('fecha_emision')))
        ws.cell(row=fila, column=20, value=ref_info.get('codigo_tipo_doc', '01'))
        ws.cell(row=fila, column=21, value=ref_serie)
        ws.cell(row=fila, column=22, value=ref_numero)

    # Extracción incompleta: dice el tipo de documento de identidad pero no capturó el número
    tipo_doc_id = factura.get('tipo_doc_identidad_receptor')
    incompleta = bool(tipo_doc_id) and str(tipo_doc_id) != '0' and not numero_doc_id

    # Riesgo conocido: emisor persona natural + receptor empresa -- patron que
    # la extraccion confunde de forma sistematica (ver sospecha_emisor_receptor_invertido)
    riesgo_inversion = sospecha_emisor_receptor_invertido(factura)

    if factura.get('requiere_revision_tributaria', True) or incompleta or riesgo_inversion:
        for col in range(1, 23):
            ws.cell(row=fila, column=col).fill = FILL_AMARILLO

print('escribir_fila_venta OK')

escribir_fila_venta OK


In [7]:
from collections import Counter

FILA_INICIO_VENTAS = 12


def periodo_de_fecha(fecha_str) -> str:
    if not fecha_str:
        return datetime.now().strftime('%Y%m')
    return str(fecha_str)[:7].replace('-', '')


def generar_registro_ventas(ventas_grupo: list[dict], ruc_emisor: str, razon_social: str,
                             periodo: str, lookup_ref: dict) -> Path:
    """Genera un Registro de Ventas (Formato 14.1) para UNA empresa (ruc_emisor) y UN periodo (mes)."""
    output_ventas = OUTPUT_DIR / f'Registro_Ventas_{ruc_emisor}_{periodo}.xlsx'
    wb_ventas = openpyxl.load_workbook(str(PLANTILLA_VENTAS))
    ws_ventas = wb_ventas.active

    ultimo_correlativo_v = get_ultimo_correlativo(ws_ventas, FILA_INICIO_VENTAS)
    docs_existentes_v = leer_docs_existentes(ws_ventas, FILA_INICIO_VENTAS, col_serie=5, col_numero=6, col_total=17)

    fila_actual_v = get_primera_fila_libre(ws_ventas, FILA_INICIO_VENTAS)
    correlativo_v = ultimo_correlativo_v + 1
    escritas_v = 0
    ignoradas_v = 0

    for f in ventas_grupo:
        clave = clave_documento(f.get('serie', ''), f.get('numero', ''), f.get('total'), f.get('fecha_emision'))
        if clave in docs_existentes_v:
            print(f'    Duplicado ignorado: {clave}')
            ignoradas_v += 1
            continue
        escribir_fila_venta(ws_ventas, fila_actual_v, correlativo_v, f, lookup_ref)
        docs_existentes_v.add(clave)
        fila_actual_v += 1
        correlativo_v += 1
        escritas_v += 1

    ws_ventas['B3'] = periodo
    ws_ventas['B4'] = ruc_emisor
    ws_ventas['B5'] = razon_social

    wb_ventas.save(str(output_ventas))
    print(f'  Registro de Ventas guardado: {output_ventas}')
    print(f'  Filas escritas: {escritas_v}  Duplicados ignorados: {ignoradas_v}')
    return output_ventas


ventas = sorted([f for f in todas_facturas if f.get('tipo_operacion') == 'VENTA'],
                key=lambda x: x.get('fecha_emision', ''))

# El Registro de Ventas SUNAT es mensual: se agrupa por (ruc_emisor, periodo del mes
# de la factura), no solo por empresa. La empresa dueña del libro = el RUC emisor
# de la factura de venta (ya viene en la extracción, sin config manual por cliente).
ventas_por_grupo: dict[tuple[str, str], list[dict]] = {}
for f in ventas:
    ruc = str(f.get('ruc_emisor', '') or 'SIN_RUC').strip()
    periodo_f = periodo_de_fecha(f.get('fecha_emision'))
    ventas_por_grupo.setdefault((ruc, periodo_f), []).append(f)

empresas_distintas = {ruc for ruc, _ in ventas_por_grupo}
print(f'Empresas distintas (por RUC emisor): {len(empresas_distintas)}')
print(f'Archivos a generar (empresa x periodo): {len(ventas_por_grupo)}\n')

archivos_ventas_generados: dict[tuple[str, str], Path] = {}
for (ruc_emisor, periodo_grupo), facturas_grupo in sorted(ventas_por_grupo.items()):
    nombres = [f.get('razon_social_emisor') for f in facturas_grupo if f.get('razon_social_emisor')]
    razon_social = Counter(nombres).most_common(1)[0][0] if nombres else 'RAZON SOCIAL NO IDENTIFICADA'
    print(f'=== {razon_social} (RUC {ruc_emisor}) — periodo {periodo_grupo} — {len(facturas_grupo)} facturas ===')
    archivos_ventas_generados[(ruc_emisor, periodo_grupo)] = generar_registro_ventas(
        facturas_grupo, ruc_emisor, razon_social, periodo_grupo, lookup_ref
    )
    print()

Empresas distintas (por RUC emisor): 14
Archivos a generar (empresa x periodo): 30

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20100154308) — periodo 202401 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20100154308_202401.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20100154308) — periodo 202402 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20100154308_202402.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== PRED & ASOCIADOS SOCIEDAD ANONIMA CERRADA (RUC 20160156400) — periodo 202403 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20160156400_202403.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== ISEM PERU SRL INDUSTRIAS Y SERVICIOS ELECTRO-MECANICOS S.R.L. (RUC 20220199968) — periodo 202311 — 2 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20220199968_202311.xlsx


  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20345725976_202310.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== RIOJA MARIN JUAN FRANCISCO (RUC 20471744493) — periodo 202412 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20471744493_202412.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== PRED & ASOCIADOS SOCIEDAD ANONIMA CERRADA (RUC 20500906710) — periodo 202403 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20500906710_202403.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20600150915) — periodo 202402 — 2 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20600150915_202402.xlsx
  Filas escritas: 2  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20601509157) — periodo 202401 — 5 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601509157

  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601509157_202403.xlsx
  Filas escritas: 2  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. (RUC 20601509157) — periodo 202405 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601509157_202405.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20601509157) — periodo 202406 — 3 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601509157_202406.xlsx
  Filas escritas: 3  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20601509157) — periodo 202410 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601509157_202410.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C. (RUC 20601509157) — periodo 202411 — 3 facturas ===
  Registro de Ventas guardado: ..\data\output\Re

  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601701503_202508.xlsx
  Filas escritas: 4  Duplicados ignorados: 0

=== P & M COURIER EXPRESS S.A.C. (RUC 20601701503) — periodo 202509 — 3 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601701503_202509.xlsx
  Filas escritas: 3  Duplicados ignorados: 0

=== P & M COURIER EXPRESS S.A.C. (RUC 20601701503) — periodo 202511 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601701503_202511.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== P & M COURIER EXPRESS S.A.C. (RUC 20601701503) — periodo 202601 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20601701503_202601.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== UNION QUIA EMPRESA DE TRANSPORTE UNION QUIA S.A.C. (RUC 20607880736) — periodo 202308 — 4 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20607880736_202308.xlsx
  Filas escritas: 4  Dup

  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20607880736_202311.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== CORPORACION FAMOD S.A. (RUC 20609182653) — periodo 202311 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20609182653_202311.xlsx
  Filas escritas: 1  Duplicados ignorados: 0

=== CORPORACION FAMOD S.A. (RUC 20609182653) — periodo 202312 — 4 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20609182653_202312.xlsx
  Filas escritas: 4  Duplicados ignorados: 0

=== EFICAX SERVICIOS GENERALES S.A.C. (RUC 20610992782) — periodo 202311 — 1 facturas ===
  Registro de Ventas guardado: ..\data\output\Registro_Ventas_20610992782_202311.xlsx
  Filas escritas: 1  Duplicados ignorados: 0



## 5. Verificación de totales

In [8]:
print('=== VERIFICACION DE TOTALES ===\n')

suma_compras_json = sum(round2(f.get('total')) for f in compras)
wb_check_c = openpyxl.load_workbook(str(output_compras))
ws_check_c = wb_check_c.active
suma_compras_excel = sum(
    round2(row[19]) for row in ws_check_c.iter_rows(min_row=FILA_INICIO_COMPRAS, values_only=True)
    if row[19] is not None
)
print(f'Compras — JSON: {suma_compras_json:.2f}  Excel: {suma_compras_excel:.2f}  '
      f'{"OK" if abs(suma_compras_json - suma_compras_excel) < 0.02 else "DIFERENCIA"}')

print()
todo_ok = True
for (ruc_emisor, periodo_grupo), facturas_grupo in sorted(ventas_por_grupo.items()):
    archivo = archivos_ventas_generados[(ruc_emisor, periodo_grupo)]
    suma_json = sum(round2(f.get('total')) for f in facturas_grupo)
    wb_check_v = openpyxl.load_workbook(str(archivo))
    ws_check_v = wb_check_v.active
    suma_excel = sum(
        round2(row[16]) for row in ws_check_v.iter_rows(min_row=FILA_INICIO_VENTAS, values_only=True)
        if row[16] is not None
    )
    estado = 'OK' if abs(suma_json - suma_excel) < 0.02 else 'DIFERENCIA'
    todo_ok = todo_ok and (estado == 'OK')
    print(f'Ventas RUC {ruc_emisor} / periodo {periodo_grupo} — JSON: {suma_json:.2f}  Excel: {suma_excel:.2f}  {estado}')

print(f'\nCuadre general: {"OK — todas las empresas/periodos cuadran" if todo_ok else "REVISAR — hay diferencias"}')

n_revision = sum(1 for f in todas_facturas if f.get('requiere_revision_tributaria'))
print(f'Facturas marcadas para revision tributaria: {n_revision}')

=== VERIFICACION DE TOTALES ===

Compras — JSON: 0.00  Excel: 0.00  OK

Ventas RUC 20100154308 / periodo 202401 — JSON: 41381.89  Excel: 41381.89  OK
Ventas RUC 20100154308 / periodo 202402 — JSON: 28339.61  Excel: 28339.61  OK
Ventas RUC 20160156400 / periodo 202403 — JSON: 49750.00  Excel: 49750.00  OK
Ventas RUC 20220199968 / periodo 202311 — JSON: 221260.61  Excel: 221260.61  OK


Ventas RUC 20269180731 / periodo 202312 — JSON: 65000.00  Excel: 65000.00  OK
Ventas RUC 20304177552 / periodo 202407 — JSON: 149860.00  Excel: 149860.00  OK
Ventas RUC 20345725976 / periodo 202310 — JSON: 19446.40  Excel: 19446.40  OK
Ventas RUC 20471744493 / periodo 202412 — JSON: 13587.70  Excel: 13587.70  OK
Ventas RUC 20500906710 / periodo 202403 — JSON: 390511.87  Excel: 390511.87  OK
Ventas RUC 20600150915 / periodo 202402 — JSON: 43368.84  Excel: 43368.84  OK
Ventas RUC 20601509157 / periodo 202401 — JSON: 52027.88  Excel: 52027.88  OK
Ventas RUC 20601509157 / periodo 202402 — JSON: 52359.30  Excel: 52359.30  OK
Ventas RUC 20601509157 / periodo 202403 — JSON: 20815.91  Excel: 20815.91  OK
Ventas RUC 20601509157 / periodo 202405 — JSON: 183868.45  Excel: 183868.45  OK


Ventas RUC 20601509157 / periodo 202406 — JSON: 200700.23  Excel: 200700.23  OK
Ventas RUC 20601509157 / periodo 202410 — JSON: 6258.01  Excel: 6258.01  OK
Ventas RUC 20601509157 / periodo 202411 — JSON: 513270.09  Excel: 513270.09  OK


Ventas RUC 20601701503 / periodo 202401 — JSON: 58962.30  Excel: 58962.30  OK
Ventas RUC 20601701503 / periodo 202402 — JSON: 32460.55  Excel: 32460.55  OK
Ventas RUC 20601701503 / periodo 202507 — JSON: 363781.73  Excel: 363781.73  OK
Ventas RUC 20601701503 / periodo 202508 — JSON: 322337.30  Excel: 322337.30  OK
Ventas RUC 20601701503 / periodo 202509 — JSON: 343548.05  Excel: 343548.05  OK
Ventas RUC 20601701503 / periodo 202511 — JSON: 219912.48  Excel: 219912.48  OK


Ventas RUC 20601701503 / periodo 202601 — JSON: 231944.81  Excel: 231944.81  OK
Ventas RUC 20607880736 / periodo 202308 — JSON: 7476.84  Excel: 7476.84  OK
Ventas RUC 20607880736 / periodo 202309 — JSON: 15210.20  Excel: 15210.20  OK
Ventas RUC 20607880736 / periodo 202311 — JSON: 12460.00  Excel: 12460.00  OK
Ventas RUC 20609182653 / periodo 202311 — JSON: 147491.50  Excel: 147491.50  OK
Ventas RUC 20609182653 / periodo 202312 — JSON: 583897.11  Excel: 583897.11  OK
Ventas RUC 20610992782 / periodo 202311 — JSON: 24190.00  Excel: 24190.00  OK

Cuadre general: OK — todas las empresas/periodos cuadran
Facturas marcadas para revision tributaria: 0
